# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing vs declining content

The FlyRank research material reports that growing content was, on average, longer and younger than declining content.

**Outcome/label:** the ML target is based on the observed `trend_direction` field. This is an observed performance outcome, not a manually assigned causal label.

**Methodology question:** the group comparison can show a directional association, but it cannot establish that adding words or reducing content age causes growth. A stronger design would compare similar pages over time and include an appropriate control group.

### Finding 2 — Freshness and mature content

The research material reports stronger measured performance for some recently refreshed/more recent content groups.

**Outcome:** the reported outcomes are observed performance measures such as growth/decline status, impressions, and composite health measures.

**Methodology question:** these are observational comparisons. A before/after intervention study with comparable unrefreshed pages would be needed to make a causal refresh-impact claim.

**Audit conclusion:** both findings are useful as measured/directional evidence, but neither should be presented as causal proof.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### My Model Under an Honest Split

The Week-5 model is evaluated again using a grouped train/test split by `client_id`.
This prevents the same client from appearing in both training and testing data.

The original Week-5 result is treated as the "before" result, while the grouped
client split is the "after" result.

The comparison uses the same target, feature set, model type, and precision@20
and precision@50 metrics. This makes the comparison more meaningful because the
evaluation metric remains unchanged.

The grouped split is more conservative because content from an unseen client is
used for testing. Therefore, the resulting score is a better estimate of how the
model may perform on clients not represented during training.

The results are treated as measured validation evidence, not proof of real-world
causal performance.

In [4]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

FEATURE_COLUMNS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down")
).astype(int)

X = df[FEATURE_COLUMNS].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def evaluate_split(X_train, X_test, y_train, y_test, label):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    preds = (scores >= 0.5).astype(int)

    result = {
        "Evaluation": label,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "ROC AUC": roc_auc_score(y_test, scores),
        "Average Precision": average_precision_score(y_test, scores),
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "Base Rate": float(y_test.mean()),
    }
    return model, scores, preds, result

# BEFORE: ordinary row-level split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
random_model, random_scores, random_preds, random_result = evaluate_split(
    X_train_r, X_test_r, y_train_r, y_test_r, "Before — random row split"
)

# AFTER: grouped split by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g = X.iloc[train_idx].copy()
X_test_g = X.iloc[test_idx].copy()
y_train_g = y.iloc[train_idx].copy()
y_test_g = y.iloc[test_idx].copy()

grouped_model, grouped_scores, grouped_preds, grouped_result = evaluate_split(
    X_train_g, X_test_g, y_train_g, y_test_g, "After — client-grouped split"
)

comparison = pd.DataFrame([random_result, grouped_result])
display(comparison.round(3))

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Grouped training rows:", len(train_idx))
print("Grouped testing rows:", len(test_idx))
print("Grouped training clients:", len(train_clients))
print("Grouped testing clients:", len(test_clients))
print("Client overlap:", train_clients.intersection(test_clients))

assert train_clients.isdisjoint(test_clients)
assert comparison["Precision@20"].between(0, 1).all()
assert comparison["Precision@50"].between(0, 1).all()

print("\nML-09 split verification: PASS")


,Evaluation,Precision@20,Precision@50,ROC AUC,Average Precision,Accuracy,Precision,Recall,F1,Base Rate
0,Before — random row split,1.0,1.0,0.913,0.935,0.820,0.843,0.822,0.832,0.542
1,After — client-grouped split,1.0,1.0,0.850,0.873,0.756,0.807,0.687,0.742,0.511


Grouped training rows: 23837
Grouped testing rows: 6163
Grouped training clients: 25
Grouped testing clients: 7
Client overlap: set()

ML-09 split verification: PASS


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

The final model feature set was checked against fields that could directly reveal the
target or identify the same client across the train and test sets.

The target `is_declining_label` is derived from `trend_direction`, so neither
`trend_direction` nor `trend_pct` is used as a model feature.

The identifiers `content_id` and `client_id` are also excluded from the feature set.
`client_id` is used only for the grouped train/test split.

**Direct label/identifier leakage check: PASS:** no label-derived field, target field,
or ID is included as a model feature.

**Temporal-overlap limitation:** several performance features (including `last_30d`
and `90d` measures) describe observed performance windows that may overlap the period
used to define `trend_direction`. Therefore, the PASS above is specifically a
**direct leakage/identifier check**, not proof that the evaluation is a fully
future-safe forecasting design. The results should be interpreted as measured
decision-support evidence rather than a guaranteed future prediction.

In [5]:

# Detailed audit of the grouped evaluation
grouped_error_mask = grouped_preds != y_test_g.to_numpy()
grouped_error_count = int(grouped_error_mask.sum())
grouped_error_rate = float(grouped_error_mask.mean())

grouped_top20_idx = np.argsort(-grouped_scores)[:20]
grouped_top50_idx = np.argsort(-grouped_scores)[:50]

top20_positive = int(y_test_g.iloc[grouped_top20_idx].sum())
top50_positive = int(y_test_g.iloc[grouped_top50_idx].sum())

base_rate = float(y_test_g.mean())
p20 = grouped_result["Precision@20"]
p50 = grouped_result["Precision@50"]

print("Grouped test base rate:", round(base_rate, 3))
print("Precision@20:", round(p20, 3), f"({top20_positive}/20 positives)")
print("Precision@50:", round(p50, 3), f"({top50_positive}/50 positives)")
print("ROC AUC:", round(grouped_result["ROC AUC"], 3))
print("Average Precision:", round(grouped_result["Average Precision"], 3))
print("Accuracy:", round(grouped_result["Accuracy"], 3))
print("Precision:", round(grouped_result["Precision"], 3))
print("Recall:", round(grouped_result["Recall"], 3))
print("F1:", round(grouped_result["F1"], 3))
print("Classification errors at threshold 0.5:", grouped_error_count)
print("Classification error rate:", round(grouped_error_rate, 3))

lift20 = p20 / base_rate if base_rate else np.nan
lift50 = p50 / base_rate if base_rate else np.nan

print("Precision@20 lift over base rate:", round(lift20, 2), "x")
print("Precision@50 lift over base rate:", round(lift50, 2), "x")

assert grouped_error_count >= 0
assert 0 <= grouped_error_rate <= 1
assert abs(p20 - top20_positive / 20) < 1e-12
assert abs(p50 - top50_positive / 50) < 1e-12

error_summary = pd.DataFrame({
    "Metric": [
        "Test base rate",
        "Precision@20",
        "Precision@50",
        "ROC AUC",
        "Average Precision",
        "Accuracy",
        "Precision at 0.5",
        "Recall at 0.5",
        "F1 at 0.5",
        "Classification errors",
    ],
    "Value": [
        base_rate, p20, p50,
        grouped_result["ROC AUC"],
        grouped_result["Average Precision"],
        grouped_result["Accuracy"],
        grouped_result["Precision"],
        grouped_result["Recall"],
        grouped_result["F1"],
        grouped_error_count,
    ],
})

display(error_summary.round(3))


Grouped test base rate: 0.511
Precision@20: 1.0 (20/20 positives)
Precision@50: 1.0 (50/50 positives)
ROC AUC: 0.85
Average Precision: 0.873
Accuracy: 0.756
Precision: 0.807
Recall: 0.687
F1: 0.742
Classification errors at threshold 0.5: 1503
Classification error rate: 0.244
Precision@20 lift over base rate: 1.96 x
Precision@50 lift over base rate: 1.96 x


,Metric,Value
0,Test base rate,0.511
1,Precision@20,1.000
2,Precision@50,1.000
3,ROC AUC,0.850
4,Average Precision,0.873
5,Accuracy,0.756
6,Precision at 0.5,0.807
7,Recall at 0.5,0.687
8,F1 at 0.5,0.742
9,Classification errors,1503.000


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Logistic Regression model accurately predicts which webpages are declining and
is better than the rule-based baseline.

### Safer rewritten claim

On this anonymized dataset and the evaluated client-grouped test split, the Logistic
Regression model measured Precision@20 of 1.00 and Precision@50 of 1.00, **outperforming
the Week-4 baseline on those metrics** (0.55 and 0.48 respectively). These results
provide directional decision-support evidence that the selected historical features
can separate the observed declining and non-declining groups in this evaluation.

This result should not be interpreted as proof of causal performance or guaranteed
performance on new clients, future data, or other datasets.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.